# Room amenity preprocessing

## Read data

In [1]:
import pandas as pd
import os
import math

In [2]:
df = pd.read_csv("../../data/processed/room_amenities.csv")
df

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
12042,12043,"['Hướng Núi', 'Máy sấy tóc', 'Vòi sen', 'Wi-Fi..."
12043,12044,"['Hướng Núi', 'Máy sấy tóc', 'Vòi sen', 'Wi-Fi..."
12044,12045,"['Wi-Fi [miễn phí]', 'Điều hòa', '20 m²', 'Tối..."
12045,12046,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ..."


## Classify the unstructured and structured row

In [3]:
import ast
def classify(value):
    # --- NULL / rỗng ---
    if pd.isna(value):
        return "null"
    s = str(value).strip()
    if s == "" or s.lower() == "null" or s == "[]":
        return "null"
    
    # --- Try to parse string to object Python ---
    try:
        obj = ast.literal_eval(s)
    except Exception:
        # Have data but cannot parse  -> unstructured
        return "unstructured"
    
    # --- amenity + type ---
    if isinstance(obj, list) and len(obj) > 0 and all(isinstance(x, dict) for x in obj):
        has_amenity = all('amenity' in x for x in obj)
        has_type = all('type' in x for x in obj)
        if has_amenity and has_type:
            return "structured"
    
    # --- unstructured ---
    return "unstructured"

# classify
df["amenity_group"] = df["room_amenities"].apply(classify)

# 3. display result
counts = df["amenity_group"].value_counts()
total = len(df)

print(counts, "\n")
print("Số dòng có dạng amenity/type rõ ràng (structured):", counts.get("structured", 0))
print("Số dòng có dữ liệu nhưng không đúng dạng (unstructured):", counts.get("unstructured", 0))
print("Số dòng NULL / rỗng (null):", counts.get("null", 0))
print("\nTổng số dòng:", total)

amenity_group
structured      7927
unstructured    4010
null             110
Name: count, dtype: int64 

Số dòng có dạng amenity/type rõ ràng (structured): 7927
Số dòng có dữ liệu nhưng không đúng dạng (unstructured): 4010
Số dòng NULL / rỗng (null): 110

Tổng số dòng: 12047


In [4]:
# Divide into 3 groups

df_structured = df[df["amenity_group"] == "structured"].copy()
df_unstructured = df[df["amenity_group"] == "unstructured"].copy()
df_null        = df[df["amenity_group"] == "null"].copy()

print("Structured:", df_structured.shape)
print("Unstructured:", df_unstructured.shape)
print("Null:", df_null.shape)


Structured: (7927, 3)
Unstructured: (4010, 3)
Null: (110, 3)


In [5]:
df_structured

,room_type_id,room_amenities,amenity_group
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",structured
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':...",structured
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':...",structured
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':...",structured
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':...",structured
...,...,...,...
12022,12023,"[{'amenity': 'Diện tích phòng: 55 m²', 'type':...",structured
12027,12028,"[{'amenity': 'Diện tích phòng: 15 m²', 'type':...",structured
12028,12029,"[{'amenity': 'Diện tích phòng: 12 m²', 'type':...",structured
12029,12030,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':...",structured


In [6]:
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"['Hướng Ngoài trời', 'phòng tắm riêng', 'Điều ...",unstructured
296,297,"['2 phòng ngủ', '2 phòng tắm', 'Máy sấy tóc', ...",unstructured
297,298,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured
298,299,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured
299,300,"['Tối đa 2 người lớn', '1 giường đôi lớn']",unstructured
...,...,...,...
12042,12043,"['Hướng Núi', 'Máy sấy tóc', 'Vòi sen', 'Wi-Fi...",unstructured
12043,12044,"['Hướng Núi', 'Máy sấy tóc', 'Vòi sen', 'Wi-Fi...",unstructured
12044,12045,"['Wi-Fi [miễn phí]', 'Điều hòa', '20 m²', 'Tối...",unstructured
12045,12046,"['Hướng Ngoài trời', 'Ấm nước điện', 'phòng tắ...",unstructured


In [7]:
df_null

,room_type_id,room_amenities,amenity_group
37,38,[],null
40,41,[],null
587,588,[],null
588,589,[],null
888,889,[],null
...,...,...,...
11709,11710,[],null
11710,11711,[],null
11717,11718,[],null
11758,11759,[],null


## Check for each type of amenities

In [8]:
import ast
from collections import Counter, defaultdict
import pandas as pd

types = []
# dict: type -> list amenity
type_to_amenities = defaultdict(list)

for s in df_structured["room_amenities"]:
    # parse chuỗi thành list[dict]
    obj = ast.literal_eval(str(s))
    for item in obj:
        t = item.get("type")
        a = item.get("amenity")
        if pd.notna(t) and t != "NULL":
            t = str(t)
            types.append(t)
            if pd.notna(a):
                type_to_amenities[t].append(str(a))

counter = Counter(types)

print("Tổng số loại type khác nhau:", len(counter))
print("\nTổng quan từng type:")
print("(type, số lần xuất hiện, số amenity khác nhau)\n")

for t, c in counter.most_common():
    distinct_amen = len(set(type_to_amenities[t]))
    print(f"{t:25s} {c:5d}  |  {distinct_amen:4d} amenity khác nhau")

# =============================
# List each AMENITY for each TYPE
# =============================

print("\n\nCHI TIẾT TỪNG TYPE:\n")

for t, c in counter.most_common():       # search each type
    amen_counter = Counter(type_to_amenities[t])
    print(f"=== TYPE: {t} ===")
    print(f"Tổng số lần xuất hiện: {c}")
    print(f"Số amenity khác nhau: {len(amen_counter)}")
    print("Danh sách amenity:")

    for amen, cnt in amen_counter.most_common():
        print(f"  - {amen}  --> {cnt}")
    print()   # \n


Tổng số loại type khác nhau: 37

Tổng quan từng type:
(type, số lần xuất hiện, số amenity khác nhau)

confirmation-instant       9441  |    14 amenity khác nhau
bed                        7625  |   713 amenity khác nhau
sqm                        7244  |   190 amenity khác nhau
bathrooms                  6277  |    17 amenity khác nhau
views                      5455  |    25 amenity khác nhau
balcony-terrace            3366  |     1 amenity khác nhau
non-smoking-room           3282  |     1 amenity khác nhau
closet                     3016  |     1 amenity khác nhau
air-conditioning           2977  |     1 amenity khác nhau
mini-bar                   2613  |     1 amenity khác nhau
blackout-curtains          2466  |     1 amenity khác nhau
hair-dryer                 1416  |     1 amenity khác nhau
bedroom                    1288  |    15 amenity khác nhau
extra-long-beds            1140  |     1 amenity khác nhau
complimentary-bottled-water  1096  |     1 amenity khác nhau
bathtub    

**Some key word**:
- bed: giường
- extra-long-beds: giường cực dài
- sqm: m2
- bathrooms: phòng tắm
- views: hướng
- balcony-terrace: Ban công/sân hiên
- non-smoking-room: Không hút thuốc
- closet: Tủ
- air-conditioning: Điều hòa
- mini-bar: Tủ lạnh nhỏ trong phòng
- blackout-curtains: Rèm che ánh sáng
- hair-dryer: Máy sấy tóc
- extra-long-beds: Giường cực dài
- complimentary-bottled-water: Nước đóng chai miễn phí
- bathtub: Bồn tắm
- bedroom: phòng ngủ
- shower: Vòi sen
- separate-shower-and-tub: Bồn tắm/vòi sen riêng
- refrigerator: Tủ lạnh
- high-floor: tầng cao
- dressing-room: Phòng thay đồ
- ground-floor: tầng trệt
- private-pool: Bể bơi riêng
- executive-lounge-access: Được vào phòng chờ Thương Gia
- top-floor: tầng thượng
- complimentary-instant-coffee: Cà phê hòa tan miễn phí
- jacuzzi-bathtub: Bồn tắm tạo sóng
- smoking-allowed: Cho phép hút thuốc
- electric-blanket: Chăn điện
- free-welcome-drink: Đồ uống mời khách miễn phí
- low-floor: tầng thấp
- wifi: Truy cập Internet - không dây/ Wi-Fi [tính phí]
- complimentary-tea: Trà miễn phí
- coffee-tea-maker: Máy pha trà/cà phê
- air-bath-access: bồn tắm lộ thiên
- hot-spring-access: vào suối nước nóng
- internet: Internet mạng LAN trong phòng [miễn phí]/ Truy cập Internet - mạng LAN

# Mapping

In [9]:
import ast
import unicodedata
import pandas as pd

# remove sign + tolower -> keyword
def normalize(text: str) -> str:
    s = unicodedata.normalize("NFKD", str(text))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().replace("đ", "d")

# type -> list keyword 
TYPE_KEYWORDS = {
    "bed": [
        "giuong",          
        "nem futon",       
        "bunk bed",
    ],
    "sqm": [
        "dien tich phong", 
        "m2",
        "m²",
    ],
    "bathrooms": [
        "phong tam",       
        "wc",
        "nha ve sinh",
    ],
    "views": [
        "huong thanh pho",
        "huong ngoai troi",
        "huong bien",
        "huong nui",
        "view",
        "huong ho",
        "huong vuon",
        "huong"
    ],
    "balcony-terrace": [
        "ban cong",
        "san hien",
        "ban cong/san hien",
    ],
    "non-smoking-room": [
        "khong hut thuoc",
        "phong khong hut thuoc",
    ],
    "closet": [
        "tu quan ao",
        "tu ao quan",
    ],
    "air-conditioning": [
        "dieu hoa",
        "may lanh",
        "air conditioning",
    ],
    "mini-bar": [
        "tu lanh nho",
        "mini bar",
        "minibar",
    ],
    "blackout-curtains": [
        "rem che anh sang",
        "rem chan sang",
    ],
    "hair-dryer": [
        "may say toc",
        "hair dryer",
    ],
    "extra-long-beds": [
        "giuong cuc dai",
        "giuong sieu dai",
        "extra long bed",
    ],
    "complimentary-bottled-water": [
        "nuoc dong chai mien phi",
        "bottled water",
    ],
    "bathtub": [
        "bon tam ",
        "bon tắm ",   
        "bathtub",
        "bon"
    ],
    "bedroom": [
        "phong ngu",
        "phong ngủ",
        "bedroom",
        "studio/1 phong ngu",
    ],
    "shower": [
        "voi sen",
        "shower",
    ],
    "separate-shower-and-tub": [
        "bon tam/voi sen rieng",
        "bon tam/vòi sen rieng",
        "separate shower and tub",
    ],
    "refrigerator": [
        "tu lanh",
        "refrigerator",
    ],
    "high-floor": [
        "tang cao",
        "high floor",
    ],
    "dressing-room": [
        "phong thay do",
        "dressing room",
    ],
    "ground-floor": [
        "tang tret",
        "tang triệt",
        "ground floor",
    ],
    "private-pool": [
        "be boi rieng",
        "ho boi rieng",
        "private pool",
    ],
    "executive-lounge-access": [
        "phong cho thuong gia",
        "executive lounge",
    ],
    "top-floor": [
        "tang thuong",
        "top floor",
    ],
    "complimentary-instant-coffee": [
        "ca phe hoa tan mien phi",
        "instant coffee mien phi",
    ],
    "jacuzzi-bathtub": [
        "bon tam tao song",
        "jacuzzi",
    ],
    "smoking-allowed": [
        "cho phep hut thuoc",
        "smoking allowed",
    ],
    "electric-blanket": [
        "chan dien",
        "electric blanket",
    ],
    "free-welcome-drink": [
        "do uong moi khach mien phi",
        "welcome drink mien phi",
    ],
    "low-floor": [
        "tang thap",
        "low floor",
    ],
    "wifi": [
        "wi-fi",
        "wifi",
        "truy cap internet - khong day",
    ],
    "complimentary-tea": [
        "tra mien phi",
        "complimentary tea",
    ],
    "coffee-tea-maker": [
        "may pha tra/ca phe",
        "coffee/tea maker",
    ],
    "air-bath-access": [
        "bon tam lo thien",
        "air bath",
    ],
    "hot-spring-access": [
        "suoi nuoc nong",
        "hot spring",
    ],
    "internet": [
        "internet mang lan",
        "mang lan",
        "internet ",
    ],
}


In [10]:
def infer_type_from_amenity(amenity: str):
    """
    Nhận chuỗi amenity (Tiếng Việt) và trả về type (str) hoặc None nếu không đoán được.
    """
    if not isinstance(amenity, str):
        return None

    text = normalize(amenity)

    matched_types = []

    for t, keywords in TYPE_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                matched_types.append(t)
                break  

    if not matched_types:
        return None

    # If match >1 type, use this prio (just in case):
    priority = [
        "sqm", "bed", "bathrooms", "bedroom",
        "views", "air-conditioning", "refrigerator",
    ]
    for t in priority:
        if t in matched_types:
            return t

    # No prio type, use 1st type
    return matched_types[0]


In [11]:
import math

def map_room_amenities_cell(cell):
    """
    cell: chuỗi dạng list (JSON-like) chứa amenity.
    Trả về list các dict: {"amenity": ..., "type": ...}
    """
    if pd.isna(cell) or (isinstance(cell, float) and math.isnan(cell)):
        return cell

    # Parse string -> list Python
    text = str(cell)
    try:
        objs = ast.literal_eval(text)
    except Exception:
        # fail to parse -> cell = 1 amenity string
        objs = [text]

    # if not list -> convert to list
    if not isinstance(objs, (list, tuple)):
        objs = [objs]

    new_objs = []

    for item in objs:
        if isinstance(item, dict):
            amen = item.get("amenity")
            typ = item.get("type")

            # If type = NULL / empty -> guess more
            if (typ is None) or (isinstance(typ, str) and typ.strip().upper() in ("", "NULL")):
                guessed = infer_type_from_amenity(amen)
                if guessed is not None:
                    item["type"] = guessed

            new_objs.append(item)

        else:
            amen = str(item)
            guessed = infer_type_from_amenity(amen)
            new_objs.append({
                "amenity": amen,
                "type": guessed,   # None if cannot guess
            })

    return new_objs

In [12]:
# map the unstructed
df_unstructured["room_amenities"] = (
    df_unstructured["room_amenities"].apply(map_room_amenities_cell)
)
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}...",unstructured
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': Non...",unstructured
...,...,...,...
12042,12043,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'...",unstructured
12043,12044,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'...",unstructured
12044,12045,"[{'amenity': 'Wi-Fi [miễn phí]', 'type': 'wifi...",unstructured
12045,12046,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured


**Checking the unstructed**

In [13]:
import ast
from collections import Counter, defaultdict
import pandas as pd

types = []
# dict: type -> list amenity
type_to_amenities = defaultdict(list)

for s in df_unstructured["room_amenities"]:
    # parse chuỗi thành list[dict]
    obj = ast.literal_eval(str(s))
    for item in obj:
        t = item.get("type")
        a = item.get("amenity")
        if pd.notna(t) and t != "NULL":
            t = str(t)
            types.append(t)
            if pd.notna(a):
                type_to_amenities[t].append(str(a))

counter = Counter(types)

print("Tổng số loại type khác nhau:", len(counter))
print("\nTổng quan từng type:")
print("(type, số lần xuất hiện, số amenity khác nhau)\n")

for t, c in counter.most_common():
    distinct_amen = len(set(type_to_amenities[t]))
    print(f"{t:25s} {c:5d}  |  {distinct_amen:4d} amenity khác nhau")

# =============================
# List each AMENITY for each TYPE
# =============================

print("\n\nCHI TIẾT TỪNG TYPE:\n")

for t, c in counter.most_common():       # search each type
    amen_counter = Counter(type_to_amenities[t])
    print(f"=== TYPE: {t} ===")
    print(f"Tổng số lần xuất hiện: {c}")
    print(f"Số amenity khác nhau: {len(amen_counter)}")
    print("Danh sách amenity:")

    for amen, cnt in amen_counter.most_common():
        print(f"  - {amen}  --> {cnt}")
    print()   # \n

Tổng số loại type khác nhau: 29

Tổng quan từng type:
(type, số lần xuất hiện, số amenity khác nhau)

bathrooms                  4537  |    18 amenity khác nhau
bed                        3942  |   368 amenity khác nhau
sqm                        3558  |   114 amenity khác nhau
hair-dryer                 2966  |     1 amenity khác nhau
views                      2607  |    25 amenity khác nhau
air-conditioning           2563  |     2 amenity khác nhau
refrigerator               2499  |     2 amenity khác nhau
wifi                       1929  |     2 amenity khác nhau
shower                     1873  |     1 amenity khác nhau
complimentary-bottled-water  1356  |     1 amenity khác nhau
closet                     1128  |     1 amenity khác nhau
balcony-terrace             980  |     1 amenity khác nhau
blackout-curtains           962  |     1 amenity khác nhau
bathtub                     741  |     4 amenity khác nhau
complimentary-instant-coffee   588  |     1 amenity khác nhau
bedroom 

In [14]:
# df_unstructured.to_csv(
#     "room_unstructured_mapped.csv",
#     index=False,
#     encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
# )

In [15]:
explode_unstructured = df_unstructured.explode("room_amenities")
print(explode_unstructured)

       room_type_id                                     room_amenities  \
295             296   {'amenity': 'Hướng Ngoài trời', 'type': 'views'}   
295             296  {'amenity': 'phòng tắm riêng', 'type': 'bathro...   
295             296  {'amenity': 'Điều hòa', 'type': 'air-condition...   
295             296                {'amenity': '25 m²', 'type': 'sqm'}   
295             296    {'amenity': 'Tối đa 2 người lớn', 'type': None}   
...             ...                                                ...   
12046         12047  {'amenity': 'Điều hòa', 'type': 'air-condition...   
12046         12047        {'amenity': 'Tủ quần áo', 'type': 'closet'}   
12046         12047                {'amenity': '20 m²', 'type': 'sqm'}   
12046         12047    {'amenity': 'Tối đa 4 người lớn', 'type': None}   
12046         12047     {'amenity': '2 giường đôi lớn', 'type': 'bed'}   

      amenity_group  
295    unstructured  
295    unstructured  
295    unstructured  
295    unstructured  
2

In [16]:
explode_unstructured["type"] = explode_unstructured["room_amenities"].apply(lambda x: x.get("type") if isinstance(x, dict) else None)
explode_unstructured["amenity"] = explode_unstructured["room_amenities"].apply(lambda x: x.get("amenity") if isinstance(x, dict) else None)

In [17]:
print(explode_unstructured["type"])

295                 views
295             bathrooms
295      air-conditioning
295                   sqm
295                  None
               ...       
12046    air-conditioning
12046              closet
12046                 sqm
12046                None
12046                 bed
Name: type, Length: 42498, dtype: object


In [18]:
none_type_df = explode_unstructured[explode_unstructured["type"].isna()]

In [19]:
print(none_type_df)

       room_type_id                                     room_amenities  \
295             296    {'amenity': 'Tối đa 2 người lớn', 'type': None}   
296             297    {'amenity': 'Tối đa 4 người lớn', 'type': None}   
297             298          {'amenity': 'Ấm nước điện', 'type': None}   
297             298  {'amenity': 'Đồ dùng cho giấc ngủ thoải mái', ...   
297             298    {'amenity': 'Trái cây/đồ ăn vặt', 'type': None}   
...             ...                                                ...   
12044         12045    {'amenity': 'Tối đa 4 người lớn', 'type': None}   
12045         12046          {'amenity': 'Ấm nước điện', 'type': None}   
12045         12046    {'amenity': 'Tối đa 2 người lớn', 'type': None}   
12046         12047          {'amenity': 'Ấm nước điện', 'type': None}   
12046         12047    {'amenity': 'Tối đa 4 người lớn', 'type': None}   

      amenity_group  type                         amenity  
295    unstructured  None              Tối đa 2 ngư

In [20]:
# none_type_df.to_csv(
#     "room_unstructured_none_type.csv",
#     index=False,
#     encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
# )

In [21]:
explode_unstructured.loc[explode_unstructured["type"].isna(), ["amenity"]].drop_duplicates().sort_values("amenity").to_csv(
    "room_unstructured_none_type_amenity.csv",index=False,)

In [22]:
none_type = pd.read_csv("room_unstructured_none_type_amenity.csv")
none_type["normalized_amenity"] = none_type["amenity"].apply(normalize)
print(none_type)

                            amenity              normalized_amenity
0                            Cửa sổ                          cua so
1               Cửa sổ có thể mở ra             cua so co the mo ra
2                         Cửa sổ mở                       cua so mo
3                   Hành lang ngoài                 hanh lang ngoai
4            Lối vào bãi biển riêng          loi vao bai bien rieng
..                              ...                             ...
160              Tối đa 9 người lớn              toi da 9 nguoi lon
161  Đồ dùng cho giấc ngủ thoải mái  do dung cho giac ngu thoai mai
162                Đồ gỗ ngoài trời                do go ngoai troi
163                         Ấm nước                         am nuoc
164                    Ấm nước điện                    am nuoc dien

[165 rows x 2 columns]


In [23]:
df_unstructured

,room_type_id,room_amenities,amenity_group
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}...",unstructured
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': Non...",unstructured
...,...,...,...
12042,12043,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'...",unstructured
12043,12044,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'...",unstructured
12044,12045,"[{'amenity': 'Wi-Fi [miễn phí]', 'type': 'wifi...",unstructured
12045,12046,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view...",unstructured


In [24]:
# mapping bổ sung cho TYPE_KEYWORDS
#  --- IGNORE ---
TYPE_KEYWORDS.update(
    {
        "confirmation-instant" : [
        "hanh lang ngoai" ,
        "loi vao bai bien rieng"  ,
        "may ca phe espresso co vien nen (co phi)"  ,
        "phong xong kho"  ,
        "quat"  ,
        "ruou"  ,
        "suoi"  ,
        "trai cay/do an vat"  ,
        "do dung cho giac ngu thoai mai"  ,
        "do go ngoai troi"  ,
        "am nuoc"  ,
        "am nuoc dien" ,
        ]
    }
)

In [25]:
def infer_type_from_amenity(amenity: str):
    """
    Nhận chuỗi amenity (Tiếng Việt) và trả về type (str) hoặc None nếu không đoán được.
    """
    if not isinstance(amenity, str):
        return None

    text = normalize(amenity)

    matched_types = []

    for t, keywords in TYPE_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                matched_types.append(t)
                break  

    if not matched_types:
        return None

    # If match >1 type, use this prio (just in case):
    priority = [
        "sqm", "bed", "bathrooms", "bedroom",
        "views", "air-conditioning", "refrigerator"
    ]
    for t in priority:
        if t in matched_types:
            return t

    # No prio type, use 1st type
    return matched_types[0]


In [26]:
import math

def map_room_amenities_cell(cell):
    """
    cell: chuỗi dạng list (JSON-like) chứa amenity.
    Trả về list các dict: {"amenity": ..., "type": ...}
    """
    if pd.isna(cell) or (isinstance(cell, float) and math.isnan(cell)):
        return cell

    # Parse string -> list Python
    text = str(cell)
    try:
        objs = ast.literal_eval(text)
    except Exception:
        # fail to parse -> cell = 1 amenity string
        objs = [text]

    # if not list -> convert to list
    if not isinstance(objs, (list, tuple)):
        objs = [objs]

    new_objs = []

    for item in objs:
        if isinstance(item, dict):
            amen = item.get("amenity")
            typ = item.get("type")

            # If type = NULL / empty -> guess more
            if (typ is None) or (isinstance(typ, str) and typ.strip().upper() in ("", "NULL")):
                guessed = infer_type_from_amenity(amen)
                if guessed is not None:
                    item["type"] = guessed

            new_objs.append(item)

        else:
            amen = str(item)
            guessed = infer_type_from_amenity(amen)
            new_objs.append({
                "amenity": amen,
                "type": guessed,   # None if cannot guess
            })

    return new_objs

In [27]:
def update_room_amenities(amenities):
    for amenity in amenities:
        if amenity.get("type") is None:
            guessed = infer_type_from_amenity(amenity.get("amenity"))
            if guessed is not None:
                amenity["type"] = guessed
    return amenities

In [28]:
df_unstructured_v2 = df_unstructured.copy()
df_unstructured_v2["room_amenities"] = df_unstructured_v2["room_amenities"].apply(update_room_amenities)

In [29]:
# df_unstructured_v2.to_csv(
#     "room_unstructured_mapped_v2.csv",
#     index=False,
#     encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
# )

In [30]:
df_unstructured_v2.drop(columns=["amenity_group"], inplace=True)

In [31]:
df_unstructured_v2

,room_type_id,room_amenities
295,296,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
296,297,"[{'amenity': '2 phòng ngủ', 'type': 'bedroom'}..."
297,298,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
298,299,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."
299,300,"[{'amenity': 'Tối đa 2 người lớn', 'type': Non..."
...,...,...
12042,12043,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'..."
12043,12044,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'..."
12044,12045,"[{'amenity': 'Wi-Fi [miễn phí]', 'type': 'wifi..."
12045,12046,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."


In [32]:
df_structured.drop(columns=["amenity_group"], inplace=True)

In [33]:
df_structured

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
12022,12023,"[{'amenity': 'Diện tích phòng: 55 m²', 'type':..."
12027,12028,"[{'amenity': 'Diện tích phòng: 15 m²', 'type':..."
12028,12029,"[{'amenity': 'Diện tích phòng: 12 m²', 'type':..."
12029,12030,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."


In [34]:
merge_df = pd.concat([df_structured, df_unstructured_v2], ignore_index=True)

In [35]:
merge_df

,room_type_id,room_amenities
0,1,"[{'amenity': 'Diện tích phòng: 18 m²', 'type':..."
1,2,"[{'amenity': 'Diện tích phòng: 30 m²', 'type':..."
2,3,"[{'amenity': 'Diện tích phòng: 45 m²', 'type':..."
3,4,"[{'amenity': 'Diện tích phòng: 20 m²', 'type':..."
4,5,"[{'amenity': 'Diện tích phòng: 35 m²', 'type':..."
...,...,...
11932,12043,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'..."
11933,12044,"[{'amenity': 'Hướng Núi', 'type': 'views'}, {'..."
11934,12045,"[{'amenity': 'Wi-Fi [miễn phí]', 'type': 'wifi..."
11935,12046,"[{'amenity': 'Hướng Ngoài trời', 'type': 'view..."


In [36]:
merge_df.to_csv(
    "room_amenities_mapped.csv",
    index=False,
    encoding="utf-8-sig"   # để mở bằng Excel không lỗi dấu
)